# Gradient boosting : LightGBM et XGBoost
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Objectif : tester si le gradient boosting améliore la référence actuelle, la régression
logistique réglée (val acc 0,9293).

**Trois façons d'utiliser LightGBM et XGBoost**
1. **TF-IDF** : mêmes n-grammes de caractères et mots que la régression logistique
   (matrice creuse de grande dimension).
2. **Caractéristiques construites** (une trentaine) : pour chaque mot et chaque terminaison
   (2, 3, 4 lettres), proportion d'hommes observée dans le train, lissée vers la moyenne.
   Ces valeurs sont agrégées sur les mots du nom (max, min, moyenne, mot le plus informatif,
   somme des log-odds), ce qui les rend **indépendantes de l'ordre NOM / PRÉNOMS**.
   Sur le train, elles sont calculées **hors pli** (validation croisée 5 plis) pour éviter
   que chaque nom « voie » sa propre étiquette.
3. **Empilement (stacking)** : caractéristiques construites + probabilité de la régression
   logistique (elle aussi hors pli sur le train).

**Protocole** : découpage figé commun à tous les notebooks ; réglages choisis sur la
**validation**. Aucun score n'est calculé sur le test ici : avec `RUN_TEST`, les
probabilités val + test des meilleurs modèles sont enregistrées pour
`evaluation_finale.ipynb`.

## 0. Paramètres

In [ ]:
RUN_TFIDF    = False   # 1. boosting sur TF-IDF (très lent, désactivé par défaut)
RUN_FEATURES = True    # 2. boosting sur caractéristiques construites
RUN_STACKING = True    # 3. empilement avec la régression logistique
N_TRIALS     = 40      # essais Optuna par modèle (caractéristiques construites, stacking)
N_TRIALS_TFIDF = 10    # essais Optuna par modèle sur TF-IDF (plus lents)
RUNS_DIR     = 'runs'
MLFLOW_URI   = 'sqlite:///mlflow.db'
RUN_TEST     = False   # enregistre les prédictions val + test des meilleurs modèles
FINAL_DIR    = 'runs/final'

## 1. Configuration

In [ ]:
import os, re, json, time, unicodedata, logging, warnings
from collections import Counter

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
import mlflow
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

warnings.filterwarnings('ignore', category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('mlflow').setLevel(logging.WARNING)
logging.getLogger('alembic').setLevel(logging.WARNING)

DATASET_PATH = 'dataset_nettoye.csv'
# 4 threads : sous WSL, LightGBM et XGBoost se bloquent avec 12 threads (testé),
# et 4 est le plus rapide sur cette machine
N_JOBS       = min(4, os.cpu_count())
os.makedirs(RUNS_DIR, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('boosting')
print(f"LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | {N_JOBS} coeurs")

## 2. Données et découpage

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r"['-]", ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(DATASET_PATH)
df['nom_clean']     = df['NOM'].apply(clean_text)
df['prenoms_clean'] = df['PRENOMS'].fillna('').apply(clean_text)
df['full_name']     = (df['nom_clean'] + ' ' + df['prenoms_clean']).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label'] = (df['GENRE'] == 'M').astype(int)

# Découpage FIGÉ (split_reference.csv), commun à tous les notebooks
SPLIT_PATH = 'split_reference.csv'

def load_split(df):
    import hashlib
    if not os.path.exists(SPLIT_PATH):
        raise FileNotFoundError(f"{SPLIT_PATH} introuvable : il fige le découpage train/val/test")
    sp = pd.read_csv(SPLIT_PATH)
    emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
    assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(), \
        f"{SPLIT_PATH} ne correspond pas au dataset (ordre ou contenu des noms modifié)"
    return [df.index[sp['ensemble'].values == e] for e in ('train', 'val', 'test')]

idx_tr, idx_va, idx_te = load_split(df)
tr_texts, va_texts, te_texts = (list(df.loc[i, 'full_name']) for i in (idx_tr, idx_va, idx_te))
y_tr, y_va = df.loc[idx_tr, 'label'].values, df.loc[idx_va, 'label'].values
print(f"Train={len(tr_texts):,} | Val={len(va_texts):,} | test : prédictions enregistrées seulement si RUN_TEST")


def metrics(y, p):
    yp = (p > 0.5).astype(int)
    return dict(acc=float(accuracy_score(y, yp)),
                f1=float(f1_score(y, yp, average='macro')),
                auc=float(roc_auc_score(y, p)))

RESULTS, VAL_PREDS, MODELS = {}, {}, {}

def record(name, p_va, params=None, extra=None):
    """Enregistre un modèle : métriques de validation, MLflow, prédictions."""
    m = metrics(y_va, p_va)
    RESULTS[name], VAL_PREDS[name] = m, np.asarray(p_va)
    with mlflow.start_run(run_name=name):
        if params: mlflow.log_params(params)
        mlflow.log_metrics({f'final_val_{k}': v for k, v in m.items()})
        if extra: mlflow.log_metrics(extra)
    print(f"{name:<40} val acc={m['acc']:.4f} f1={m['f1']:.4f} auc={m['auc']:.4f}")
    return m

## 3. Référence : régression logistique

Configuration choisie sur la validation dans `optimisation.ipynb` : n-grammes de
caractères 1 à 6 + mots entiers, C = 2.

In [ ]:
def make_lr():
    return make_pipeline(
        make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True),
                   TfidfVectorizer(analyzer='word', token_pattern=r'\S+')),
        LogisticRegression(C=2, max_iter=3000))

lr = make_lr().fit(tr_texts, y_tr)
p_lr_va = lr.predict_proba(va_texts)[:, 1]
record('regression_logistique', p_lr_va, dict(ngram='1-6', C=2))

## 4. Outils communs : recherche Optuna pour LightGBM et XGBoost

Arrêt anticipé sur la validation (100 tours sans amélioration de la log-loss).

In [ ]:
def fit_lgb(params, X_tr, X_va):
    m = lgb.LGBMClassifier(n_estimators=3000, n_jobs=N_JOBS, verbose=-1, random_state=42, **params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='binary_logloss',
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m, m.predict_proba(X_va)[:, 1], m.best_iteration_


def fit_xgb(params, X_tr, X_va):
    m = xgb.XGBClassifier(n_estimators=3000, n_jobs=N_JOBS, tree_method='hist', random_state=42,
                          early_stopping_rounds=100, eval_metric='logloss', **params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return m, m.predict_proba(X_va)[:, 1], m.best_iteration


def lgb_space(t):
    return dict(learning_rate=t.suggest_float('learning_rate', 0.01, 0.2, log=True),
                num_leaves=t.suggest_int('num_leaves', 8, 128, log=True),
                min_child_samples=t.suggest_int('min_child_samples', 5, 100, log=True),
                colsample_bytree=t.suggest_float('colsample_bytree', 0.3, 1.0),
                subsample=t.suggest_float('subsample', 0.5, 1.0), subsample_freq=1,
                reg_lambda=t.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                reg_alpha=t.suggest_float('reg_alpha', 1e-8, 10.0, log=True))


def xgb_space(t):
    return dict(learning_rate=t.suggest_float('learning_rate', 0.01, 0.2, log=True),
                max_depth=t.suggest_int('max_depth', 3, 10),
                min_child_weight=t.suggest_float('min_child_weight', 0.5, 20.0, log=True),
                colsample_bytree=t.suggest_float('colsample_bytree', 0.3, 1.0),
                subsample=t.suggest_float('subsample', 0.5, 1.0),
                reg_lambda=t.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                reg_alpha=t.suggest_float('reg_alpha', 1e-8, 10.0, log=True))


def search(name, fit, space, X_tr, X_va, n_trials):
    """Recherche Optuna sur la validation ; enregistre le meilleur modèle."""
    def objective(t):
        _, p, n_iter = fit(space(t), X_tr, X_va)
        t.set_user_attr('n_iter', int(n_iter))
        return metrics(y_va, p)['acc']

    t0 = time.time()
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials)
    model, p_va, n_iter = fit(study.best_params, X_tr, X_va)
    MODELS[name] = model
    record(name, p_va, study.best_params,
           dict(n_iter=n_iter, n_trials=n_trials, minutes=(time.time() - t0) / 60))
    return model

## 5. Boosting sur TF-IDF

Même représentation que la régression logistique, limitée aux 50 000 n-grammes les plus
fréquents (présents au moins 2 fois) pour garder des temps raisonnables.

In [ ]:
if RUN_TFIDF:
    vec = make_union(
        TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True,
                        min_df=2, max_features=50000),
        TfidfVectorizer(analyzer='word', token_pattern=r'\S+', min_df=2))
    T_tr = vec.fit_transform(tr_texts).astype(np.float32).tocsr()
    T_va = vec.transform(va_texts).astype(np.float32).tocsr()
    print(f"TF-IDF : {T_tr.shape[1]:,} colonnes")
    search('lightgbm_tfidf', fit_lgb, lgb_space, T_tr, T_va, N_TRIALS_TFIDF)
    search('xgboost_tfidf', fit_xgb, xgb_space, T_tr, T_va, N_TRIALS_TFIDF)

## 6. Caractéristiques construites (indépendantes de l'ordre des mots)

Pour une clé (mot entier ou terminaison de k lettres), proportion lissée d'hommes dans le
train : `p = (nb_hommes + a × p0) / (nb + a)`, avec `p0` la proportion globale et `a = 2`.
Pour chaque nom, on agrège sur ses mots : max, min, moyenne, moyenne pondérée par la
fréquence, valeur du mot le plus informatif, somme des log-odds, nombre de mots connus.

In [ ]:
SUFFIXES = (2, 3, 4)
ALPHA = 2.0


def keys(word, k):
    return word if k == 0 else word[-k:]


def fit_stats(texts, labels):
    """Effectifs (total, hommes) par mot (k=0) et par terminaison de k lettres."""
    stats = {}
    for k in (0,) + SUFFIXES:
        n, pos = Counter(), Counter()
        for t, y in zip(texts, labels):
            for key in {keys(w, k) for w in t.split()}:
                n[key] += 1; pos[key] += y
        stats[k] = (n, pos)
    return stats


def featurize(texts, stats, p0):
    rows = []
    lo0 = np.log(p0 / (1 - p0))
    for t in texts:
        words = t.split()
        f = dict(n_mots=len(words), longueur=len(t))
        for k in (0,) + SUFFIXES:
            n, pos = stats[k]
            ks = [keys(w, k) for w in words]
            cnt = np.array([n.get(x, 0) for x in ks], dtype=float)
            p = (np.array([pos.get(x, 0) for x in ks]) + ALPHA * p0) / (cnt + ALPHA)
            w = np.log1p(cnt)
            info = np.abs(p - 0.5) * w
            i = int(info.argmax())
            pre = 'mot' if k == 0 else f'suf{k}'
            f.update({f'{pre}_max': p.max(), f'{pre}_min': p.min(), f'{pre}_moy': p.mean(),
                      f'{pre}_moy_pond': (p * w).sum() / w.sum() if w.sum() > 0 else p0,
                      f'{pre}_plus_info': p[i], f'{pre}_plus_info_n': w[i],
                      f'{pre}_logodds': float((np.log(p / (1 - p)) - lo0).sum()),
                      f'{pre}_connus': int((cnt > 0).sum())})
        rows.append(f)
    return pd.DataFrame(rows)


def oof_features(texts, labels, n_splits=5):
    """Caractéristiques du train calculées hors pli (pas de fuite d'étiquette)."""
    texts, labels = np.array(texts, dtype=object), np.asarray(labels)
    out = [None] * len(texts)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for fit_idx, pred_idx in skf.split(texts, labels):
        st = fit_stats(texts[fit_idx], labels[fit_idx])
        F = featurize(texts[pred_idx], st, labels[fit_idx].mean())
        for j, i in enumerate(pred_idx):
            out[i] = F.iloc[j]
    return pd.DataFrame(out).reset_index(drop=True)


if RUN_FEATURES or RUN_STACKING:
    t0 = time.time()
    F_tr = oof_features(tr_texts, y_tr)
    F_va = featurize(va_texts, fit_stats(tr_texts, y_tr), y_tr.mean())
    print(f"{F_tr.shape[1]} caractéristiques construites en {time.time()-t0:.0f} s")

In [ ]:
if RUN_FEATURES:
    lgb_feat = search('lightgbm_caracteristiques', fit_lgb, lgb_space, F_tr, F_va, N_TRIALS)
    xgb_feat = search('xgboost_caracteristiques', fit_xgb, xgb_space, F_tr, F_va, N_TRIALS)
    imp = pd.Series(lgb_feat.booster_.feature_importance('gain'), index=F_tr.columns)
    display((imp / imp.sum()).sort_values(ascending=False).head(12).round(3)
            .rename('importance (gain)').to_frame())

## 7. Empilement (stacking) avec la régression logistique

La probabilité de la régression logistique est ajoutée aux caractéristiques. Sur le train,
elle est calculée hors pli (5 plis) ; sur la validation, par la régression entraînée sur
tout le train.

In [ ]:
if RUN_STACKING:
    t0 = time.time()
    oof_lr = np.zeros(len(tr_texts))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    arr = np.array(tr_texts, dtype=object)
    for fit_idx, pred_idx in skf.split(arr, y_tr):
        oof_lr[pred_idx] = make_lr().fit(arr[fit_idx], y_tr[fit_idx]).predict_proba(arr[pred_idx])[:, 1]
    S_tr, S_va = F_tr.assign(proba_lr=oof_lr), F_va.assign(proba_lr=p_lr_va)
    print(f"Probabilités hors pli de la régression logistique : {time.time()-t0:.0f} s")
    search('lightgbm_stacking', fit_lgb, lgb_space, S_tr, S_va, N_TRIALS)
    search('xgboost_stacking', fit_xgb, xgb_space, S_tr, S_va, N_TRIALS)

## 8. Synthèse sur la validation

Écart avec la régression logistique et test de McNemar sur la **validation** (indicatif :
les réglages ont été choisis sur cette même validation, l'évaluation définitive se fera sur
le test). Ensembles simples : moyenne avec la régression logistique.

In [ ]:
from scipy.stats import binomtest

def mcnemar(p_a, p_b, y):
    e_a, e_b = (p_a > 0.5) != y, (p_b > 0.5) != y
    b, c = int((e_a & ~e_b).sum()), int((~e_a & e_b).sum())
    return b, c, float(binomtest(min(b, c), b + c, 0.5).pvalue) if b + c else 1.0

for name in [n for n in list(VAL_PREDS) if n != 'regression_logistique']:
    RESULTS[f'moyenne_lr+{name}'] = metrics(y_va, (VAL_PREDS[name] + p_lr_va) / 2)

rows = []
for name, m in RESULTS.items():
    row = dict(modele=name, **m, ecart_vs_lr=m['acc'] - RESULTS['regression_logistique']['acc'])
    if name in VAL_PREDS and name != 'regression_logistique':
        b, c, p = mcnemar(VAL_PREDS[name], p_lr_va, y_va)
        row.update(faux_seul=b, juste_seul=c, mcnemar_p=p)
    rows.append(row)
synthese = pd.DataFrame(rows).sort_values('acc', ascending=False).round(4)
display(synthese)

synthese.to_csv(os.path.join(RUNS_DIR, 'boosting_synthese.csv'), index=False)
np.savez(os.path.join(RUNS_DIR, 'boosting_val_preds.npz'), y_va=y_va, **VAL_PREDS)
print(f"Résultats enregistrés dans {RUNS_DIR}/ (probabilités uniquement, aucun nom)")

## 9. Prédictions pour l'évaluation finale (`RUN_TEST`)

Probabilités sur la validation et le test des meilleurs modèles, enregistrées dans
`runs/final/`. Aucun score de test n'est calculé ici.

In [ ]:
if RUN_TEST:
    os.makedirs(FINAL_DIR, exist_ok=True)
    if RUN_FEATURES or RUN_STACKING:
        F_te = featurize(te_texts, fit_stats(tr_texts, y_tr), y_tr.mean())
    for name, model in MODELS.items():
        if name.endswith('_caracteristiques'):
            X_te = F_te
        elif name.endswith('_stacking'):
            X_te = F_te.assign(proba_lr=lr.predict_proba(te_texts)[:, 1])
        else:
            X_te = vec.transform(te_texts).astype(np.float32).tocsr()
        np.savez(os.path.join(FINAL_DIR, f'{name}.npz'),
                 val=VAL_PREDS[name], test=model.predict_proba(X_te)[:, 1])
        print(f"{name} : prédictions enregistrées")